In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pinecone import ServerlessSpec, Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder


In [ ]:
# Load environment variables searching parent directories
load_dotenv(find_dotenv())
api_key = os.getenv('PINECONE_API_KEY')

In [17]:
import time
pc = Pinecone(api_key=api_key)
index_name = "hybrid-search"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Wait for serverless index to be fully provisioned and active
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)


In [18]:
index = pc.Index(index_name)

In [19]:
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in environment variables. Ensure your .env file is loaded and contains HF_TOKEN.")
os.environ["HF_TOKEN"] = hf_token

emmbed = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
emmbed

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4534.72it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [20]:
bm25 = BM25Encoder().default()
bm25

In [21]:
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

In [22]:
bm25.fit(sentences)
bm25.dump("bm25_values.json")
bm25 = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 2975.39it/s]


In [23]:
ret = PineconeHybridSearchRetriever(embeddings=emmbed, sparse_encoder=bm25, index=index)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11290.04it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [24]:
# Upsert documents into hybrid retriever using the sentences list
ret.add_texts(sentences)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: Index.upsert() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given

In [ ]:
ret.invoke("which city i visit most")